# Implementation

In [33]:
import torch
import torch.nn as nn


class PatchEmbedding(nn.Module):

    def __init__(
        self,
        in_channels=3,          # RGB image => 3 channels
        patch_size=16,          # Each patch is 16x16
        embedding_dim=768       # ViT Base uses 768-dim embeddings
    ):
        super().__init__()

        # Conv2D performs:
        # 1. Patch extraction
        # 2. Linear projection
        #
        # kernel_size = patch size
        # stride = patch size
        #
        # This ensures non-overlapping patches

        self.projection = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embedding_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):

        # ----------------------------
        # Input Shape
        # (B,3,224,224)
        # ----------------------------

        x = self.projection(x)

        # ----------------------------
        # Shape becomes:
        # (B,768,14,14)
        #
        # Why?
        #
        # 224/16 = 14
        #
        # So:
        #
        # Height = 14
        # Width  = 14
        #
        # 768 channels represent
        # embedding dimension
        # ----------------------------

        x = x.flatten(start_dim=2)

        # ----------------------------
        # Shape:
        # (B,768,196)
        #
        # Because:
        #
        # 14 × 14 = 196 patches
        # ----------------------------

        x = x.transpose(1, 2)

        # ----------------------------
        # Final Shape:
        # (B,196,768)
        #
        # Transformer expects:
        #
        # (Batch, Tokens, Features)
        # ----------------------------

        return x

# Testing the Module

In [34]:
dummy_image = torch.randn(
    1,      # batch size
    3,      # RGB channels
    224,    # height
    224     # width
)

patch_embed = PatchEmbedding()

output = patch_embed(dummy_image)

print(output.shape)

torch.Size([1, 196, 768])


# Chapter 2 : CLS Token Module

## Implementation

In [35]:
import torch
import torch.nn as nn


class CLS_Token(nn.Module):

    def __init__(
        self,
        embedding_dim=768
    ):
        super().__init__()

        # ------------------------------------------------
        # Learnable CLS token
        #
        # Shape:
        # (1,1,768)
        #
        # First 1 -> batch placeholder
        # Second 1 -> one token
        # 768 -> embedding dimension
        # ------------------------------------------------

        self.cls_token = nn.Parameter(
            torch.randn(
                1,
                1,
                embedding_dim
            )
        )

    def forward(self, x):

        # -----------------------------------------
        # Input:
        #
        # (B,196,768)
        # -----------------------------------------

        batch_size = x.shape[0]

        # -----------------------------------------
        # Expand CLS token for every image
        #
        # Example:
        #
        # (1,1,768)
        #
        # becomes
        #
        # (B,1,768)
        # -----------------------------------------

        cls_tokens = self.cls_token.expand(
            batch_size,
            -1,
            -1
        )

        # -----------------------------------------
        # Concatenate along token dimension
        #
        # (B,1,768)
        #
        # +
        #
        # (B,196,768)
        #
        # =
        #
        # (B,197,768)
        # -----------------------------------------

        x = torch.cat(
            (cls_tokens, x),
            dim=1
        )

        return x

# Testing CLS Token

In [36]:
dummy_tokens = torch.randn(
    2,
    196,
    768
)

cls_layer = CLS_Token()

output = cls_layer(dummy_tokens)

print(output.shape)

torch.Size([2, 197, 768])


## Combining PatchEmbedding + CLS

In [37]:
dummy_image = torch.randn(
    2,
    3,
    224,
    224
)

patch_embed = PatchEmbedding()

tokens = patch_embed(dummy_image)

print(tokens.shape)

cls_layer = CLS_Token()

tokens = cls_layer(tokens)

print(tokens.shape)

torch.Size([2, 196, 768])
torch.Size([2, 197, 768])


# Chapter 3 : Positional Embedding Module

# IMPLEMENTATION

In [38]:
import torch
import torch.nn as nn


class PositionEmbedding(nn.Module):

    def __init__(
        self,
        num_tokens=197,          # CLS + Patch Tokens
        embedding_dim=768        # Token dimension
    ):
        super().__init__()

        # --------------------------------------------------
        # Learnable position embeddings
        #
        # Shape:
        # (1,197,768)
        #
        # 1   -> Batch placeholder
        # 197 -> Total tokens
        # 768 -> Embedding dimension
        # --------------------------------------------------

        self.position_embedding = nn.Parameter(
            torch.randn(
                1,
                num_tokens,
                embedding_dim
            )
        )

    def forward(self, x):

        # ------------------------------------------
        # Input:
        # (B,197,768)
        # ------------------------------------------

        x = x + self.position_embedding

        # ------------------------------------------
        # Output:
        # (B,197,768)
        #
        # Shape remains unchanged
        # Only position information is added
        # ------------------------------------------

        return x

## Testing Position Embeddings

In [39]:
dummy_tokens = torch.randn(
    2,
    197,
    768
)

pos_embed = PositionEmbedding()

output = pos_embed(dummy_tokens)

print(output.shape)

torch.Size([2, 197, 768])


# Building the Complete Input Pipeline

In [40]:
dummy_image = torch.randn(
    2,
    3,
    224,
    224
)

# ----------------------------------
# Step 1
# Patch Embedding
# ----------------------------------

patch_embed = PatchEmbedding()

x = patch_embed(dummy_image)

print(
    "After PatchEmbedding:",
    x.shape
)

# ----------------------------------
# Step 2
# CLS Token
# ----------------------------------

cls_layer = CLS_Token()

x = cls_layer(x)

print(
    "After CLS Token:",
    x.shape
)

# ----------------------------------
# Step 3
# Position Embedding
# ----------------------------------

pos_layer = PositionEmbedding()

x = pos_layer(x)

print(
    "After Position Embedding:",
    x.shape
)

After PatchEmbedding: torch.Size([2, 196, 768])
After CLS Token: torch.Size([2, 197, 768])
After Position Embedding: torch.Size([2, 197, 768])


# Chapter 4 : Multi-Head Self-Attention (MHSA)

# Implementation
CREATING Q,K,V LAYER

In [41]:
import torch
import torch.nn as nn


class QKVProjection(nn.Module):

    def __init__(
        self,
        embedding_dim=768
    ):
        super().__init__()

        # ---------------------------------
        # Query Projection
        #
        # Input : 768
        # Output: 768
        # ---------------------------------

        self.query = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        # ---------------------------------
        # Key Projection
        # ---------------------------------

        self.key = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        # ---------------------------------
        # Value Projection
        # ---------------------------------

        self.value = nn.Linear(
            embedding_dim,
            embedding_dim
        )

    def forward(self, x):

        Q = self.query(x)

        K = self.key(x)

        V = self.value(x)

        return Q, K, V

# Full Single-Head Attention

In [42]:
import math
import torch
import torch.nn as nn


class SelfAttention(nn.Module):

    def __init__(
        self,
        embedding_dim=768
    ):
        super().__init__()

        self.query = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.key = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.value = nn.Linear(
            embedding_dim,
            embedding_dim
        )

    def forward(self, x):

        # ----------------------------------
        # Input:
        # (B,197,768)
        # ----------------------------------

        Q = self.query(x)

        # (B,197,768)

        K = self.key(x)

        # (B,197,768)

        V = self.value(x)

        # (B,197,768)

        scores = torch.matmul(
            Q,
            K.transpose(-2,-1)
        )

        # ----------------------------------
        # Shape:
        # (B,197,197)
        # ----------------------------------

        scores = scores / math.sqrt(
            K.shape[-1]
        )

        # ----------------------------------
        # Scale attention scores
        # ----------------------------------

        attention = torch.softmax(
            scores,
            dim=-1
        )

        # ----------------------------------
        # Convert scores into probabilities
        # ----------------------------------

        output = torch.matmul(
            attention,
            V
        )

        # ----------------------------------
        # Shape:
        # (B,197,768)
        # ----------------------------------

        return output

## Testing

In [43]:
dummy_tokens = torch.randn(
    2,
    197,
    768
)

attention = SelfAttention()

output = attention(dummy_tokens)

print(output.shape)

torch.Size([2, 197, 768])


# Chapter 5 : MultiHeadAttention Module (Actual ViT Implementation)

## Implementation

In [44]:
import torch
import torch.nn as nn
import math


class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        embedding_dim=768,
        num_heads=12
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.num_heads = num_heads

        self.head_dim = (
            embedding_dim // num_heads
        )

        assert (
            embedding_dim % num_heads == 0
        ), "Embedding dimension must be divisible by number of heads"

        # Generate Q, K, V together
        self.qkv = nn.Linear(
            embedding_dim,
            embedding_dim * 3
        )

        # Output projection
        self.projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )

    def forward(self, x):

        batch_size = x.shape[0]
        num_tokens = x.shape[1]

        # (B, N, 768) -> (B, N, 2304)
        qkv = self.qkv(x)

        # (B, N, 3, 12, 64)
        qkv = qkv.reshape(
            batch_size,
            num_tokens,
            3,
            self.num_heads,
            self.head_dim
        )

        # (3, B, 12, N, 64)
        qkv = qkv.permute(
            2,
            0,
            3,
            1,
            4
        )

        Q = qkv[0]
        K = qkv[1]
        V = qkv[2]

        # (B, 12, N, N)
        attention_scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        attention_scores = (
            attention_scores
            / math.sqrt(self.head_dim)
        )

        attention_weights = torch.softmax(
            attention_scores,
            dim=-1
        )

        # (B, 12, N, 64)
        attention_output = torch.matmul(
            attention_weights,
            V
        )

        # (B, N, 12, 64)
        attention_output = attention_output.permute(
            0,
            2,
            1,
            3
        )

        # (B, N, 768)
        attention_output = attention_output.reshape(
            batch_size,
            num_tokens,
            self.embedding_dim
        )

        output = self.projection(
            attention_output
        )

        return output

# Complete Forward Function

In [45]:
def forward(self, x):

    batch_size = x.shape[0]

    num_tokens = x.shape[1]

    qkv = self.qkv(x)

    qkv = qkv.reshape(
        batch_size,
        num_tokens,
        3,
        self.num_heads,
        self.head_dim
    )

    qkv = qkv.permute(
        2,
        0,
        3,
        1,
        4
    )

    Q = qkv[0]

    K = qkv[1]

    V = qkv[2]

    attention_scores = torch.matmul(
        Q,
        K.transpose(-2,-1)
    )

    attention_scores = (
        attention_scores
        /
        math.sqrt(self.head_dim)
    )

    attention_weights = torch.softmax(
        attention_scores,
        dim=-1
    )

    attention_output = torch.matmul(
        attention_weights,
        V
    )

    attention_output = (
        attention_output.permute(
            0,
            2,
            1,
            3
        )
    )

    attention_output = (
        attention_output.reshape(
            batch_size,
            num_tokens,
            self.embedding_dim
        )
    )

    output = self.projection(
        attention_output
    )

    return output

## Testing

In [47]:
dummy_tokens = torch.randn(
    2,
    197,
    768
)

mha = MultiHeadAttention()

output = mha(dummy_tokens)

print(output.shape)

torch.Size([2, 197, 768])


# Chapter 6 : MLP Block (Feed Forward Network)

# Implementation

In [48]:
import torch
import torch.nn as nn


class MLPBlock(nn.Module):

    def __init__(
        self,
        embedding_dim=768,
        mlp_ratio=4
    ):
        super().__init__()

        # ------------------------------------
        # Hidden dimension
        #
        # 768 × 4 = 3072
        # ------------------------------------

        hidden_dim = (
            embedding_dim * mlp_ratio
        )

        # ------------------------------------
        # First Linear Layer
        #
        # 768 → 3072
        # ------------------------------------

        self.fc1 = nn.Linear(
            embedding_dim,
            hidden_dim
        )

        # ------------------------------------
        # GELU Activation
        # ------------------------------------

        self.activation = nn.GELU()

        # ------------------------------------
        # Second Linear Layer
        #
        # 3072 → 768
        # ------------------------------------

        self.fc2 = nn.Linear(
            hidden_dim,
            embedding_dim
        )

    def forward(self, x):

        # ------------------------------------
        # Input:
        #
        # (B,197,768)
        # ------------------------------------

        x = self.fc1(x)

        # ------------------------------------
        # Shape:
        #
        # (B,197,3072)
        # ------------------------------------

        x = self.activation(x)

        # ------------------------------------
        # Shape remains:
        #
        # (B,197,3072)
        # ------------------------------------

        x = self.fc2(x)

        # ------------------------------------
        # Shape:
        #
        # (B,197,768)
        # ------------------------------------

        return x

# TESTING

In [49]:
dummy_tokens = torch.randn(
    2,
    197,
    768
)

mlp = MLPBlock()

output = mlp(dummy_tokens)

print(output.shape)

torch.Size([2, 197, 768])


# Chapter 7 : Transformer Encoder Block

## COMPLETE INIT..

In [51]:
class TransformerEncoderBlock(nn.Module):

    def __init__(
        self,
        embedding_dim=768,
        num_heads=12,
        mlp_ratio=4
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(
            embedding_dim
        )

        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads
        )

        self.norm2 = nn.LayerNorm(
            embedding_dim
        )

        self.mlp = MLPBlock(
            embedding_dim,
            mlp_ratio
        )

    def forward(self, x):

        residual = x

        x = self.norm1(x)

        x = self.attention(x)

        x = x + residual

        residual = x

        x = self.norm2(x)

        x = self.mlp(x)

        x = x + residual

        return x

# TESTING

In [52]:
dummy_tokens = torch.randn(
    2,
    197,
    768
)

encoder = TransformerEncoderBlock()

output = encoder(dummy_tokens)

print(output.shape)

torch.Size([2, 197, 768])


# Chapter 8 : VisionTransformer Class (Complete Model)

In [59]:
class VisionTransformer(nn.Module):

    def __init__(
        self,
        image_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=10,
        embedding_dim=768,
        num_heads=12,
        num_layers=12,
        mlp_ratio=4
    ):
        super().__init__()

        self.patch_embedding = PatchEmbedding(
            in_channels,
            patch_size,
            embedding_dim
        )

        self.num_patches = (
            image_size // patch_size
        ) ** 2

        self.cls_token = CLS_Token(
            embedding_dim
        )

        self.position_embedding = PositionEmbedding(
            self.num_patches + 1,
            embedding_dim
        )

        self.encoder_blocks = nn.ModuleList(
            [
                TransformerEncoderBlock(
                    embedding_dim,
                    num_heads,
                    mlp_ratio
                )
                for _ in range(num_layers)
            ]
        )

        self.norm = nn.LayerNorm(
            embedding_dim
        )

        self.classifier = nn.Linear(
            embedding_dim,
            num_classes
        )

    def forward(self, x):

        # -----------------------------------
        # Convert image into tokens
        # -----------------------------------

        x = self.patch_embedding(x)

        # -----------------------------------
        # Add CLS token
        # -----------------------------------

        x = self.cls_token(x)

        # -----------------------------------
        # Add positional information
        # -----------------------------------

        x = self.position_embedding(x)

        # -----------------------------------
        # Encoder stack
        # -----------------------------------

        for encoder in self.encoder_blocks:

            x = encoder(x)

        # -----------------------------------
        # Final normalization
        # -----------------------------------

        x = self.norm(x)

        # -----------------------------------
        # Extract CLS token
        # -----------------------------------

        cls_output = x[:, 0]

        # -----------------------------------
        # Classification
        # -----------------------------------

        logits = self.classifier(
            cls_output
        )

        return logits

# Testing The Complete Model

In [60]:
model = VisionTransformer(
    num_classes=10
)

dummy_image = torch.randn(
    2,
    3,
    224,
    224
)

output = model(dummy_image)

print(output.shape)

torch.Size([2, 10])
